# Week 3 · Lab Workshop
## Your first predictor
*AI for Creativity · Level 6 · work in pairs*

**Brief:** pick a creative dataset, explore and clean it, train a model, and test who it favours.

| Track | What you do |
|---|---|
| 🟢 **Core** | Fill in the blanks to load, chart, split and train a decision tree; record accuracy |
| 🔵 **Stretch** | Compare a decision tree with a random forest, chart feature importance, and check accuracy by group |

Replace every `FILL_ME` with your own value. Save outputs and screenshots to your **R&D journal**.

**How to use this notebook**

- Runs in **Google Colab** (*File → Upload notebook*) and in **JupyterLab** (Anaconda).
- Run cells **top to bottom** with **Shift + Enter**.
- Cells marked 🌐 download a pretrained model — run them in Colab (internet needed). In Colab, choose *Runtime → Change runtime type → GPU* for faster training.
- Broke something? *Runtime → Restart* (Colab) or *Kernel → Restart* (Jupyter) and run from the top.

## Setup — run this cell (dataset generators)

In [1]:
import numpy as np
import pandas as pd

def make_songs(n=600, seed=7):
    """A synthetic song dataset (invented for teaching) with a hidden bias towards pop."""
    rng = np.random.default_rng(seed)
    genre = rng.choice(["pop", "rock", "hip-hop", "khaleeji", "jazz"], n, p=[.45, .2, .2, .08, .07])
    danceability = np.clip(rng.normal(.6, .15, n) + (genre == "hip-hop") * .15 + (genre == "khaleeji") * .1, 0, 1)
    energy = np.clip(rng.normal(.6, .2, n) + (genre == "rock") * .15 - (genre == "jazz") * .2, 0, 1)
    acousticness = np.clip(rng.normal(.3, .2, n) + (genre == "jazz") * .35 + (genre == "khaleeji") * .25, 0, 1)
    tempo = rng.normal(118, 22, n).round()
    duration_min = np.clip(rng.normal(3.4, .6, n), 1.8, 7).round(2)
    # 'popular' was decided by past charts that favoured pop-style features
    score = 2.2 * danceability + 1.5 * energy - 1.2 * acousticness + (genre == "pop") * .8 + rng.normal(0, .5, n)
    popular = (score > np.quantile(score, .55)).astype(int)
    df = pd.DataFrame(dict(genre=genre, danceability=danceability.round(3), energy=energy.round(3),
                           acousticness=acousticness.round(3), tempo=tempo, duration_min=duration_min, popular=popular))
    df.loc[rng.choice(n, 12, replace=False), "tempo"] = np.nan      # a few missing values to clean
    return df

def make_dataset(kind="songs", n=500, seed=11):
    """Synthetic creative datasets for the lab: songs, films, artworks or fashion (invented for teaching)."""
    rng = np.random.default_rng(seed)
    if kind == "songs":
        return make_songs(n, seed)
    if kind == "films":
        genre = rng.choice(["drama", "comedy", "action", "documentary", "animation"], n, p=[.3, .25, .25, .1, .1])
        budget_m = np.round(np.exp(rng.normal(2.5, 1, n)), 1)
        runtime = rng.normal(108, 18, n).round()
        festival = rng.integers(0, 2, n)
        rating = np.clip(rng.normal(6.4, 1.1, n) + festival * .5 + (genre == "documentary") * .4, 1, 10).round(1)
        score = .02 * budget_m + .9 * (rating - 6) + (genre == "action") * .6 + rng.normal(0, .6, n)
        label = (score > np.quantile(score, .55)).astype(int)
        return pd.DataFrame(dict(genre=genre, budget_m=budget_m, runtime=runtime, festival=festival, rating=rating, box_office_hit=label))
    if kind == "artworks":
        medium = rng.choice(["oil", "digital", "photography", "sculpture", "calligraphy"], n, p=[.3, .3, .2, .1, .1])
        width_cm = rng.normal(80, 30, n).clip(10).round()
        artist_followers_k = np.round(np.exp(rng.normal(2, 1.2, n)), 1)
        colour_saturation = rng.uniform(0, 1, n).round(3)
        score = .03 * artist_followers_k + .01 * width_cm + (medium == "oil") * .7 + rng.normal(0, .7, n)
        label = (score > np.quantile(score, .6)).astype(int)
        return pd.DataFrame(dict(medium=medium, width_cm=width_cm, artist_followers_k=artist_followers_k, colour_saturation=colour_saturation, sold_at_auction=label))
    if kind == "fashion":
        category = rng.choice(["abaya", "streetwear", "eveningwear", "sportswear", "accessories"], n, p=[.2, .3, .15, .2, .15])
        price_aed = np.round(np.exp(rng.normal(5.5, .7, n)))
        sustainable = rng.integers(0, 2, n)
        instagram_posts = rng.poisson(30, n)
        score = .02 * instagram_posts - .0004 * price_aed + sustainable * .4 + (category == "streetwear") * .5 + rng.normal(0, .5, n)
        label = (score > np.quantile(score, .5)).astype(int)
        return pd.DataFrame(dict(category=category, price_aed=price_aed, sustainable=sustainable, instagram_posts=instagram_posts, best_seller=label))
    raise ValueError("kind must be songs, films, artworks or fashion")

## 🟢 Core · Step 1 — Choose your dataset
Options: `"songs"`, `"films"`, `"artworks"`, `"fashion"`. Using your own CSV? `df = pd.read_csv("my_data.csv")`.

In [ ]:
kind = FILL_ME                      # e.g. "fashion"
df = make_dataset(kind)
label = df.columns[-1]               # the last column is the label
print("Predicting:", label)
df.head()

## 🟢 Core · Step 2 — Clean and explore (make 2 charts)

In [ ]:
import matplotlib.pyplot as plt
df = df.fillna(df.median(numeric_only=True))
group_col = df.select_dtypes("object").columns[0]          # the category column, e.g. genre

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df[group_col].value_counts().plot.bar(ax=axes[0], title=f"Rows per {group_col}")
df.groupby(group_col)[label].mean().plot.barh(ax=axes[1], title=f"Share with {label} = 1")
plt.tight_layout(); plt.show()

## 🟢 Core · Step 3 — Split and train

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = pd.get_dummies(df.drop(columns=label), columns=[group_col])
y = df[label]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=FILL_ME, random_state=0, stratify=y)   # e.g. 0.25

model = DecisionTreeClassifier(max_depth=FILL_ME, random_state=0)     # try 2, 4, 8
model.fit(X_train, y_train)
print(f"Train {model.score(X_train, y_train):.2f} | Test {model.score(X_test, y_test):.2f}")

## 🟢 Core · Step 4 — Who does it favour?

In [ ]:
test = df.loc[X_test.index].copy()
test["predicted"] = model.predict(X_test)
test.groupby(group_col).agg(rows=(label, "size"), actual=(label, "mean"), predicted=("predicted", "mean")).round(2)

✅ **Core done.** Record: dataset, depth, train/test accuracy, and one group the model may treat unfairly.

---
## 🔵 Stretch · Random forest and feature importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = FILL_ME          # build a RandomForestClassifier with 200 trees and random_state=0, then fit it
# print train/test accuracy, then plot forest.feature_importances_ as a horizontal bar chart

---
## ✍️ R&D journal
Write 300 words: *Which feature mattered most, and what does that tell you about the people who created the labels?* Include your two charts.